# The statistical components, and how many this panel supports

*A learning exercise performed in role: a simulated mandate with no client and no institution. Nothing in this notebook is investment advice, a recommendation, or a client communication.*

**Entry point** `python3 -m portfolio_workbench.factors.components`

**Modules covered** `factors/components.py`

_Generated from the code by `python3 -m reporting.notebooks`: the module headers below are read out of the modules themselves, and the run is the entry point's own output._

## 1. What this module does, and the source of every method in it

The statistical family extracts directions of common variation from the sleeve correlation matrix and decides how many of them the panel supports. The entry point prints the eigenvalues against both references, the retention count per window, the component portfolios, and the stability check on the count.

**Sources.** Every public function of the modules this notebook covers, and what it traces to. The map is checked over the code by the acceptance fixture, so a method added without a source fails a command rather than going unnoticed.

- `factors/components.py::component_loadings` traces to principal components extraction over the sleeve correlation matrix, with the retention rule measured rather than assumed
- `factors/components.py::component_portfolios` traces to the component portfolios of this effort: fully invested weights recovered from the component series, since a standardised score's own weights sum to zero
- `factors/components.py::count_series` traces to principal components extraction over the sleeve correlation matrix, with the retention rule measured rather than assumed
- `factors/components.py::decide` traces to principal components extraction over the sleeve correlation matrix, with the retention rule measured rather than assumed
- `factors/components.py::decompose` traces to principal components extraction over the sleeve correlation matrix, with the retention rule measured rather than assumed
- `factors/components.py::eigen_structure` traces to principal components extraction over the sleeve correlation matrix, with the retention rule measured rather than assumed
- `factors/components.py::main` traces to principal components extraction over the sleeve correlation matrix, with the retention rule measured rather than assumed
- `factors/components.py::mp_edge` traces to the Marchenko-Pastur upper bulk edge, Marchenko & Pastur (1967), Matematicheskii Sbornik 114(1): `(1 + sqrt(n/t))^2` on the standardised panel
- `factors/components.py::orient` traces to principal components extraction over the sleeve correlation matrix, with the retention rule measured rather than assumed
- `factors/components.py::permutation_null` traces to the matched empirical null of this effort: independent permutation of each series, so the null is the panel's own marginals rather than a simulation assumption
- `factors/components.py::retained` traces to the retention rule of this effort, stated before it was applied: an eigenvalue beating the 95th percentile of the matched null, with the Kaiser eigenvalue-above-one rule rejected and the scree and parallel-analysis heuristics recorded in the dossier
- `factors/components.py::stability` traces to the stability check of this effort: the count re-decided on permuted windows, so a count that moves with the seed is refused by its own rule
- `factors/components.py::threshold` traces to principal components extraction over the sleeve correlation matrix, with the retention rule measured rather than assumed
- `factors/components.py::varimax` traces to the varimax rotation, Kaiser (1958), Psychometrika 23(3), used only to orient an extracted basis and never to select one

## 2. Why it works this way, including what was rejected

_The module headers, verbatim: each records why the module is shaped the way it is, what was rejected, and the measurement that settled it. They are quoted here rather than restated, so the notebook cannot drift from the code._

**`factors/components.py`**

The statistical family: principal components of the panel's own returns, and the rule that
decides how many of them exist.

The count is decided by an external criterion stated in advance, never by out-of-sample
performance: choosing the count on the results it produces and then reporting it is circular.
Here the criterion is a **matched empirical null**. For each estimation window the null is
built by giving every series its own independent random circular shift, which preserves each
series' values, its marginal distribution and its autocorrelation, and destroys only the
alignment between series - the cross-correlation the test needs a null for. A plain
permutation of each series would also destroy the series' own serial dependence, making the
null less correlated in time than the panel and the threshold too low, which is the wrong
direction for a rule that decides how many factors to trust.

Two reference points are printed beside the mechanical count. The **Marchenko-Pastur edge**
`(1 + sqrt(N/T))^2` is the analytic bulk edge for an uncorrelated panel and is a cross-check on
the simulation rather than the threshold, because monthly returns are not normal and the edge
assumes they are. A **pre-registered fixed count** of three is reported as well: it was fixed before the rule was
applied, so a mechanical count that disagrees with it is a result to report rather than a number
to overwrite.

Both of these are counts on a correlation matrix, so every share of variance prints beside the
**pure-noise share** for the same panel: at eleven series and this many observations the first
two components carry about a quarter of the variance with no common structure at all, and a
share read against zero would make that quarter look like a finding.

Extraction is extraction. The components are directions of common variation in this panel's
covariance. They are not selected factors, they are not identified as priced, and their labels
are descriptive: nothing here says a component *is* a term-structure factor or a market factor.
Selection required the criterion above, and the criterion is about variance, not about premia.

**A panel this small undercounts.** Eigenvalue-based rules on a few dozen observations understate
the number of directions a longer sample would support, so the count here is a decision about this
panel's usable dimension and never a count of the factors in the market: the number is reported
with the panel it was computed on, and no result may describe it as "the number of factors".

## 3. The data contract it consumes, and the as-of rule

The decomposition runs inside the trailing sixty-month window, so the count a window retains cannot see the month it will be used to trade. The count rule is stated before it is applied: an eigenvalue beats the 95th percentile of a matched permutation null built by permuting each series independently, which keeps the panel's own marginals and its own autocorrelation structure rather than assuming normal noise. Extraction returns loadings and shares; component *series* are taken from the component portfolios, because a standardised score has mean zero by construction and a series with an invented mean is not a return.

## 4. The worked example on small numbers, with the identity checked

The Marchenko-Pastur edge and the retention rule are both hand-checkable. The worked example computes the edge for eleven series over sixty rows from its closed form, and applies the rule to three eigenvalues against a three-column null.

The cell below runs on numbers small enough to check by hand and asserts the identity, so a reader can see the arithmetic rather than take the module's word for it.

In [1]:
import numpy as np

from portfolio_workbench.factors import components

# The Marchenko-Pastur upper edge for n series over t rows, from its closed form.
edge = components.mp_edge(11, 60)
assert abs(edge - (1 + np.sqrt(11 / 60)) ** 2) < 1e-12
print(f"eleven series over sixty rows: edge {edge:.4f}")

# The rule counts observed components that beat the null's own threshold for their position.
null = np.array([[1.30, 1.10, 1.00], [1.20, 1.00, 0.90], [1.25, 1.05, 0.95]])
assert components.retained(np.array([3.00, 1.15, 0.50]), null) == 2
assert components.retained(np.array([1.00, 0.90, 0.80]), null) == 0
print("two of three observed components clear the matched null; a noise panel clears none")

eleven series over sixty rows: edge 2.0397
two of three observed components clear the matched null; a noise panel clears none


## 5. The real run: inputs, parameters, provenance block

The provenance block is printed first, then the parameters this module decides under, then the entry point's own report. The report is the module's output rather than a transcription of it, so a number quoted from a notebook is the number the module prints.

In [2]:
from portfolio_workbench.data import loader, universe

document = loader.load_panel()
months = document.months
print(f"snapshot {document.snapshot_id}, taken as of {document.as_of}")
print(f"panel {len(months)} months {months.min()}..{months.max()} across {len(universe.TICKERS)} sleeves")
print("manifest fields: " + ", ".join(sorted(document.manifest)))

from portfolio_workbench.factors import components

print(f"pre-registered count bound k <= {components.PREREGISTERED_K}, null percentile {components.PERCENTILE}")
print(f"permutation draws per window {components.DRAWS}, seed {components.SEED}")

snapshot 2026-09-13, taken as of 2026-09-13T07:56:28+00:00
panel 191 months 2010-09..2026-07 across 11 sleeves
manifest fields: created, excluded, files, instruments, snapshot_id, window
pre-registered count bound k <= 3, null percentile 95
permutation draws per window 200, seed 20260912


In [3]:
import subprocess
import sys

finished = subprocess.run(
    [sys.executable, "-m", "portfolio_workbench.factors.components"], capture_output=True, text=True, cwd="."
)
print(finished.stdout)
assert finished.returncode == 0, finished.stderr

[factor] snapshot 2026-09-13, 191 returns × 11 sleeves
[factor] full panel: Marchenko-Pastur edge 1.5376, permutation null 95th percentile of the top eigenvalue 1.7549, observed top 5.049; the matched null sits above the analytic edge, which is what keeping the panel's own marginals rather than a normal's does
[factor] the rule retains 2 component(s); the pre-registered fixed count is 3 and the eigenvalues run 5.05, 1.74, 1.17, 0.92, 0.59, 0.55
    component 1: variance share 45.90% against a pure-noise share of 12.77% at the same N and T
    component 2: variance share 15.79% against a pure-noise share of 11.59% at the same N and T
[factor] the first two components together: 61.69% of variance against 24.33% under pure noise
[factor] walk-forward, 131 windows: the mechanical count takes 1, 2; it moves by more than one component in 0.0% of steps, against the 25% that falsifies the rule
[factor] in-window references: Marchenko-Pastur edge 2.04..2.05, null 95th percentile 2.07..2.33, obs

## 6. Results, and how to read them, including the resolution limit and what a reader must not conclude

The count is the answer to how many directions this panel supports, and its resolution limit is the gap between the null's threshold and the eigenvalue it is compared with: a panel whose eigenvalues sit near the edge gives a count that moves with the window. The stability check is printed for that reason. A reader must not read the extracted components as priced factors, and must not read the rule as selecting them: extraction finds directions of common variation, and selection needs an external criterion stated in advance, which here is the retention rule and the bound on the count.

## 7. What this module does not establish

Nothing here establishes that the components are economically identified, that they are stable outside the sample, or that a count of two is the right number rather than the count this rule retains on this panel. The rule's own bound is a pre-registered constraint, not a finding about markets.